# Laboratorio: Clasificación de Imágenes con Perceptrón Multicapa (MLP) en PyTorch
## Dataset: Fashion-MNIST (10 Clases de Prendas de Vestir)

En este cuadernillo se implementa, valida, entrena y evalúa un modelo de **Perceptrón Multicapa (MLP - Multi-Layer Perceptron)** utilizando **PyTorch** para resolver una tarea de clasificación multiclase sobre el dataset **Fashion-MNIST**.

El desarrollo sigue rigurosamente:
1. **La metodología de experimentación y receta de entrenamiento** del cuadernillo `07 receta_entrenamiento.ipynb` (verificación de dimensiones, fit de una sola muestra, fit de un lote, función `fit` con validación cruzada y guardado del mejor modelo por *checkpointing*).
2. **El catálogo exhaustivo de métricas de clasificación** del cuadernillo `08 metricas_clasificacion.ipynb` (*Accuracy*, Matriz de Confusión, *Precision*, *Recall*, *F1-Score*, y Curvas *ROC* / *ROC-AUC* multiclase).
3. **La estructura ordenada y modular por etapas** del cuadernillo `02_reg_lin_mul_01_vuelos.ipynb`, donde cada fase se presenta de forma secuencial y las funciones o clases se definen estrictamente en la etapa en que se invocan por primera vez.

---

### Descripción del Problema y del Dataset
- **Dataset:** Fashion-MNIST (almacenado en formato CSV en `Dataset/fashion-mnist_train.csv` y `Dataset/fashion-mnist_test.csv`).
- **Muestras ($m$):** 60,000 imágenes para entrenamiento/validación y 10,000 imágenes para prueba independiente.
- **Características ($n$):** 784 píxeles por imagen ($28 \times 28$ píxeles en escala de grises, cada uno con valores de brillo entre 0 y 255).
- **Variable objetivo ($y$):** `label`, valor entero entre 0 y 9 que representa la categoría de prenda:
  - `0`: T-shirt/top (Camiseta / Top)
  - `1`: Trouser (Pantalón)
  - `2`: Pullover (Suéter)
  - `3`: Dress (Vestido)
  - `4`: Coat (Abrigo)
  - `5`: Sandal (Sandalia)
  - `6`: Shirt (Camisa)
  - `7`: Sneaker (Zapatilla deportiva)
  - `8`: Bag (Bolso / Cartera)
  - `9`: Ankle boot (Botín / Bota al tobillo)

## 0. Inclusión de Librerías y Configuración del Entorno

Importamos las librerías necesarias para el proyecto:
- **`os`**: Manejo de rutas y archivos del sistema.
- **`numpy`**: Computación vectorial, manipulación de matrices y cálculo matemático.
- **`pandas`**: Lectura y preprocesamiento eficiente de los archivos CSV.
- **`matplotlib.pyplot`**: Graficación de imágenes, curvas de aprendizaje y matrices de confusión.
- **`torch` y `torch.nn`**: Construcción, optimización y entrenamiento de la red neuronal en PyTorch.
- **`torch.utils.data`**: Manejo estructurado de los conjuntos de datos mediante `Dataset` y `DataLoader`.
- **`sklearn.metrics`**: Cálculo de métricas avanzadas de clasificación según el cuadernillo 08.

Adicionalmente, se fijan las semillas aleatorias para garantizar la reproducibilidad de los experimentos y se selecciona dinámicamente el dispositivo de cómputo (**GPU CUDA** si se encuentra disponible, o **CPU** en caso contrario).

In [ ]:
# Manejo de rutas y sistema operativo
import os

# Computación científica y vectorial
import numpy as np

# Manipulación de datos tabulares
import pandas as pd

# Graficación y visualización de resultados
import matplotlib.pyplot as plt
%matplotlib inline

# Framework de Deep Learning (PyTorch)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Métricas de evaluación de Scikit-Learn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    auc,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize

# Fijar semillas para reproducibilidad
SEMILLA = 42
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMILLA)

# Configuración del dispositivo (GPU si está disponible, sino CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Configuración del entorno completada exitosamente.")
print(f"Dispositivo de cómputo seleccionado: {device}")
if torch.cuda.is_available():
    print(f"Nombre de la GPU: {torch.cuda.get_device_name(0)}")

### 0.1 Definición de Rutas del Dataset

Se establecen las rutas locales relativas a los archivos CSV. Si ejecutas este cuadernillo en **Google Colab**, puedes montar tu Google Drive descomentando las líneas indicadas y ajustando la ruta correspondiente.

In [ ]:
# Si ejecutas en Google Colab con Google Drive, descomenta las siguientes líneas:
# from google.colab import drive
# drive.mount('/content/gdrive')
# RUTA_BASE = '/content/gdrive/MyDrive/datasets/'

# Rutas locales relativas (carpeta Dataset en el mismo directorio del proyecto)
RUTA_TRAIN = 'Dataset/fashion-mnist_train.csv'
RUTA_TEST = 'Dataset/fashion-mnist_test.csv'

print(f"Ruta de entrenamiento configurada: {RUTA_TRAIN}")
print(f"Ruta de prueba configurada:        {RUTA_TEST}")

## 1. Lectura y Exploración del Dataset (Pandas y Matplotlib)

En esta primera etapa de la receta:
1. Cargamos los archivos `fashion-mnist_train.csv` y `fashion-mnist_test.csv` utilizando **Pandas**.
2. Verificamos la forma (*shape*) de los datos: deben contener 785 columnas (columna `label` + 784 píxeles).
3. Definimos los nombres de las 10 clases de prendas en español.
4. Exploramos las primeras filas de los datos para comprender su estructura.

In [ ]:
print("1. Cargando conjunto de entrenamiento desde CSV...")
df_train = pd.read_csv(RUTA_TRAIN)
print(f"   - Dimensiones de df_train: {df_train.shape} ({df_train.shape[0]:,} filas, {df_train.shape[1]} columnas)")

print("2. Cargando conjunto de prueba desde CSV...")
df_test = pd.read_csv(RUTA_TEST)
print(f"   - Dimensiones de df_test:  {df_test.shape} ({df_test.shape[0]:,} filas, {df_test.shape[1]} columnas)")

# Definición del diccionario y lista de clases de Fashion-MNIST en español
nombres_clases = [
    'Camiseta / Top',
    'Pantalón',
    'Suéter',
    'Vestido',
    'Abrigo',
    'Sandalia',
    'Camisa',
    'Zapatilla deportiva',
    'Bolso / Cartera',
    'Botín al tobillo'
]

clases_dict = {i: nombre for i, nombre in enumerate(nombres_clases)}

print("\nPrimeras 5 filas del conjunto de entrenamiento:")
df_train.head()

### 1.1 Distribución de Clases y Estadísticos Básicos

Es fundamental responder a las preguntas iniciales de la receta del docente:
- **¿Cuál es el tipo de problema?** Clasificación multiclase con 10 categorías exclusivas.
- **¿Están las clases balanceadas?** Verificamos el conteo de muestras por clase tanto en entrenamiento como en prueba.
- **¿Cuál es el rango de los valores de píxeles?** Comprobamos el mínimo, máximo, media y desviación estándar de las características numéricas.

In [ ]:
# Distribución de clases en el conjunto de entrenamiento
conteo_train = df_train['label'].value_counts().sort_index()
conteo_test = df_test['label'].value_counts().sort_index()

df_distribucion = pd.DataFrame({
    'Clase (ID)': range(10),
    'Nombre de Prenda': nombres_clases,
    'Muestras Train': conteo_train.values,
    'Muestras Test': conteo_test.values
})
print("Distribución de clases en ambos conjuntos:")
print(df_distribucion.to_string(index=False))

# Estadísticos de los píxeles en el conjunto de entrenamiento
pixeles_train = df_train.drop(columns=['label']).to_numpy()
print(f"\nEstadísticos de píxeles (sin normalizar):")
print(f"  - Valor Mínimo: {pixeles_train.min()}")
print(f"  - Valor Máximo: {pixeles_train.max()}")
print(f"  - Valor Medio:  {pixeles_train.mean():.4f}")
print(f"  - Desv. Est.:   {pixeles_train.std():.4f}")

# Gráfico de barras de la distribución de clases en Train
plt.figure(figsize=(10, 4))
plt.bar(range(10), conteo_train.values, color='steelblue', edgecolor='black', alpha=0.85)
plt.xticks(range(10), nombres_clases, rotation=35, ha='right', fontsize=9)
plt.ylabel('Número de Muestras')
plt.title('Distribución de Clases en el Conjunto de Entrenamiento (Fashion-MNIST)')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 1.2 Visualización de Imágenes de Muestra

Cada imagen en Fashion-MNIST está almacenada como un vector plano de 784 píxeles. Para visualizarla, redimensionamos el vector a una matriz bidimensional de $28 \times 28$ píxeles:
$$ x = i \cdot 28 + j, \quad i, j \in [0, 27] $$

Definimos la función `visualizar_muestras_dataset` para inspeccionar visualmente ejemplos de cada una de las prendas de ropa.

In [ ]:
def visualizar_muestras_dataset(df, n_filas=2, n_cols=5):
    """
    Visualiza una cuadrícula de imágenes aleatorias del DataFrame de Fashion-MNIST.
    """
    fig, axes = plt.subplots(n_filas, n_cols, figsize=(12, 5))
    indices_aleatorios = np.random.choice(len(df), n_filas * n_cols, replace=False)
    
    for idx, ax in zip(indices_aleatorios, axes.flat):
        fila = df.iloc[idx]
        etiqueta = int(fila['label'])
        pixeles = fila.drop('label').values.astype(np.uint8).reshape(28, 28)
        
        ax.imshow(pixeles, cmap='gray')
        ax.set_title(f"{nombres_clases[etiqueta]} (ID: {etiqueta})", fontsize=10, fontweight='bold')
        ax.axis('off')
        
    plt.suptitle('Muestras de Prendas del Dataset Fashion-MNIST', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Visualizar 10 prendas aleatorias
visualizar_muestras_dataset(df_train, n_filas=2, n_cols=5)

## 2. Preprocesamiento y División de Datos

Siguiendo las mejores prácticas revisadas en clases:
1. **Separación de características y etiquetas**:
   - $X$: matriz de 784 píxeles por muestra.
   - $y$: vector de etiquetas enteras (0 a 9).
2. **Normalización Min-Max**:
   - Dividimos los valores de los píxeles entre $255.0$ para escalarlos al rango continuo $[0.0, 1.0]$. Esta transformación acelera la convergencia de los gradientes y estabiliza el entrenamiento de la red neuronal.
3. **División en Entrenamiento (Train) y Validación (Val)**:
   - Siguiendo la receta del docente (`07 receta_entrenamiento.ipynb`), separamos el dataset de entrenamiento en **50,000 muestras para entrenar** y **10,000 muestras para validar**.
   - El conjunto de **Prueba (Test)** de 10,000 muestras se mantiene completamente aislado hasta la evaluación final para evitar fuga de información (*data leakage*).
4. **Construcción del `Dataset` y `DataLoader` de PyTorch**:
   - Se define la clase personalizada `FashionMNISTDataset` heredando de `torch.utils.data.Dataset`.
   - Se configuran los `DataLoader` con un tamaño de lote (*batch size*) de 64 muestras.

In [ ]:
# 1. Separar características (X) y etiquetas (y) como matrices NumPy
X_train_raw = df_train.drop(columns=['label']).to_numpy(dtype=np.float32)
y_train_raw = df_train['label'].to_numpy(dtype=np.int64)

X_test_raw = df_test.drop(columns=['label']).to_numpy(dtype=np.float32)
y_test_raw = df_test['label'].to_numpy(dtype=np.int64)

# 2. Normalización de píxeles al rango [0.0, 1.0]
X_train_norm = X_train_raw / 255.0
X_test_norm = X_test_raw / 255.0

# 3. División estratificada / aleatoria de Train en Entrenamiento (50,000) y Validación (10,000)
n_total_train = len(X_train_norm)
indices = np.random.permutation(n_total_train)

n_val = 10000
idx_val = indices[:n_val]
idx_train = indices[n_val:]

X_train, y_train = X_train_norm[idx_train], y_train_raw[idx_train]
X_val, y_val = X_train_norm[idx_val], y_train_raw[idx_val]
X_test, y_test = X_test_norm, y_test_raw

print(f"División de los datos completada:")
print(f"  - Conjunto de Entrenamiento (Train):      {X_train.shape[0]:,} muestras, {X_train.shape[1]} características")
print(f"  - Conjunto de Validación (Val):          {X_val.shape[0]:,} muestras, {X_val.shape[1]} características")
print(f"  - Conjunto de Prueba Independiente (Test): {X_test.shape[0]:,} muestras, {X_test.shape[1]} características")

# 4. Clase personalizada Dataset para PyTorch (definida en esta etapa)
class FashionMNISTDataset(Dataset):
    """
    Clase personalizada de Dataset para PyTorch basada en la receta del docente.
    Convierte arrays NumPy a tensores PyTorch y maneja el indexado.
    """
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Creación de instancias de Dataset
ds_train = FashionMNISTDataset(X_train, y_train)
ds_val = FashionMNISTDataset(X_val, y_val)
ds_test = FashionMNISTDataset(X_test, y_test)

# Configuración de DataLoaders
BATCH_SIZE = 64
dataloaders = {
    'train': DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True),
    'val':   DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False),
    'test':  DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False)
}

print(f"\nDataLoaders configurados con batch_size = {BATCH_SIZE}:")
print(f"  - Batches en Train: {len(dataloaders['train'])}")
print(f"  - Batches en Val:   {len(dataloaders['val'])}")
print(f"  - Batches en Test:  {len(dataloaders['test'])}")

## 3. Definición y Verificación de la Arquitectura MLP (Receta del Docente)

El Perceptrón Multicapa (**MLP**) es una red neuronal feedforward completamente conectada (*Fully Connected*). Para Fashion-MNIST, diseñamos la siguiente arquitectura:
- **Capa de entrada:** $D_{in} = 784$ neuronas (un valor por píxel).
- **Primera capa oculta:** $H_1 = 256$ neuronas + *Batch Normalization* + *ReLU* + *Dropout* ($p=0.2$).
- **Segunda capa oculta:** $H_2 = 128$ neuronas + *Batch Normalization* + *ReLU* + *Dropout* ($p=0.2$).
- **Capa de salida:** $D_{out} = 10$ neuronas lineales que producen los *logits* para cada una de las 10 clases de prendas.

> **Importante (Receta del Docente):** Antes de entrenar la red con todos los datos, realizamos tres verificaciones obligatorias:
> 1. **Test de dimensiones:** Verificar que al pasar un tensor con forma `(64, 784)`, la red devuelva exactamente `(64, 10)`.
> 2. **Fit de una muestra (Sanity Check 1):** Comprobar que la red es capaz de sobreajustar (*memorizar*) una sola muestra, reduciendo el loss a cero y alcanzando 100% de precisión.
> 3. **Fit de un batch (Sanity Check 2):** Comprobar que la red puede sobreajustar un lote completo de 64 muestras.

In [ ]:
def build_model(D_in=784, H1=256, H2=128, D_out=10, dropout_p=0.2):
    """
    Construye un Perceptrón Multicapa (MLP) utilizando torch.nn.Sequential,
    siguiendo la arquitectura y técnicas de regularización vistas en el cuadernillo 07.
    """
    modelo = nn.Sequential(
        # Primera capa oculta
        nn.Linear(D_in, H1),
        nn.BatchNorm1d(H1),
        nn.ReLU(),
        nn.Dropout(dropout_p),
        
        # Segunda capa oculta
        nn.Linear(H1, H2),
        nn.BatchNorm1d(H2),
        nn.ReLU(),
        nn.Dropout(dropout_p),
        
        # Capa de salida (10 logits lineales)
        nn.Linear(H2, D_out)
    ).to(device)
    return modelo

# Instanciar modelo de prueba
modelo_prueba = build_model()
print("Estructura de la red neuronal MLP:")
print(modelo_prueba)

# Verificación de dimensiones (Sanity Check 0)
tensor_prueba_input = torch.randn(64, 784).to(device)
tensor_prueba_output = modelo_prueba(tensor_prueba_input)
print(f"\nVerificación de dimensiones con tensor aleatorio:")
print(f"  - Entrada: {tensor_prueba_input.shape}  ->  (batch_size=64, n_features=784)")
print(f"  - Salida:  {tensor_prueba_output.shape}   ->  (batch_size=64, n_clases=10)")
assert tensor_prueba_output.shape == (64, 10), "Error: Las dimensiones de salida no coinciden con (64, 10)."
print("¡Verificación de dimensiones superada con éxito!")

### 3.1 Fit de una sola muestra (Sanity Check 1)

Como explica el docente: *«Una vez nos hemos asegurado de que las dimensiones de nuestra red están bien, el siguiente paso es asegurarnos de que es capaz de aprenderse de memoria una sola muestra del dataset. Si este no es el caso, podemos estar seguros de que algo no anda bien en el código o en los gradientes»*.

In [ ]:
# Tomamos la primera muestra del conjunto de entrenamiento
ds_una_muestra = FashionMNISTDataset(X_train[:1], y_train[:1])
loader_una_muestra = DataLoader(ds_una_muestra, batch_size=1)

# Instanciamos un modelo simple sin BatchNorm para 1 sola muestra
modelo_single = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

criterio_single = nn.CrossEntropyLoss()
optimizador_single = torch.optim.Adam(modelo_single.parameters(), lr=0.01)

print(f"Entrenando con una sola muestra (Etiqueta real: {y_train[0]} - {nombres_clases[y_train[0]]}):")
for epoca in range(1, 11):
    for x_b, y_b in loader_una_muestra:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizador_single.zero_grad()
        y_pred = modelo_single(x_b)
        loss = criterio_single(y_pred, y_b)
        loss.backward()
        optimizador_single.step()
        
        pred_clase = torch.argmax(y_pred, dim=1).item()
        print(f"  Época {epoca:2d}/10 | Loss: {loss.item():.6f} | Predicción: {pred_clase} ({nombres_clases[pred_clase]})")

print("\n¡Sanity Check 1 superado: el modelo memorizó perfectamente una muestra!")

### 3.2 Fit de un solo *batch* (Sanity Check 2)

El siguiente paso consiste en verificar que la red completa (con Batch Normalization y Dropout) sea capaz de memorizar un lote completo de 64 imágenes. La pérdida debe descender rápidamente hacia cero y la precisión alcanzar el 100%.

In [ ]:
# Tomamos un lote de 64 muestras
ds_un_batch = FashionMNISTDataset(X_train[:64], y_train[:64])
loader_un_batch = DataLoader(ds_un_batch, batch_size=64)

modelo_batch = build_model(dropout_p=0.0) # Sin dropout para permitir memorización perfecta
criterio_batch = nn.CrossEntropyLoss()
optimizador_batch = torch.optim.Adam(modelo_batch.parameters(), lr=0.005)

print("Entrenando sobre un solo batch (64 muestras) durante 30 épocas:")
for epoca in range(1, 31):
    modelo_batch.train()
    for x_b, y_b in loader_un_batch:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizador_batch.zero_grad()
        y_pred = modelo_batch(x_b)
        loss = criterio_batch(y_pred, y_b)
        loss.backward()
        optimizador_batch.step()
        
        pred_clases = torch.argmax(y_pred, dim=1)
        acc = (pred_clases == y_b).float().mean().item()
        
    if epoca % 5 == 0 or epoca == 1:
        print(f"  Época {epoca:2d}/30 | Pérdida (Loss): {loss.item():.5f} | Precisión (Acc): {acc * 100:.2f}%")

print("\n¡Sanity Check 2 superado: la arquitectura converge y sobreajusta el batch sin problemas!")

## 4. Entrenamiento del Modelo MLP con Conjunto de Validación

En esta etapa central implementamos el entrenamiento completo:
1. **Función de activación Softmax**:
   $$ P(y = c \mid x) = \frac{e^{z_c}}{\sum_{k=1}^{K} e^{z_k}} $$
2. **Función de costo (*Loss Function*)**:
   - `torch.nn.CrossEntropyLoss()`: Combina de forma numéricamente estable `LogSoftmax` y `NLLLoss` (entropía cruzada categórica).
3. **Optimizador**:
   - `torch.optim.Adam` con tasa de aprendizaje inicial $\alpha = 0.001$ y regularización L2 (`weight_decay = 1e-4`).
4. **Planificador de Learning Rate (*Scheduler*)**:
   - `torch.optim.lr_scheduler.ReduceLROnPlateau`: Reduce la tasa de aprendizaje si la pérdida de validación deja de mejorar durante 2 épocas consecutivas.
5. **Función `fit(...)` completa**:
   - Calcula métricas tanto para entrenamiento como para validación en cada época.
   - Implementa **Guardado del mejor modelo (*Model Checkpoint*)** cuando la precisión de validación alcanza un nuevo máximo.
   - Implementa **Parada temprana (*Early Stopping*)** para evitar sobreajuste innecesario.
   - Restaura automáticamente los mejores pesos al finalizar.

In [ ]:
def softmax(x):
    """Calcula las probabilidades de clase a partir de los logits."""
    return torch.exp(x) / torch.exp(x).sum(dim=-1, keepdim=True)

def fit(modelo, dataloaders, optimizador, criterio, scheduler=None, 
        epocas=20, early_stopping=5, nombre_archivo_ckpt='mejor_modelo_mlp.pt', verbose=1):
    """
    Bucle de entrenamiento profesional basado en la receta del docente (cuadernillo 07).
    Registra pérdidas, precisiones y learning rate por época; guarda y restaura el mejor modelo.
    """
    historia = {
        'epoca': [],
        'loss_train': [],
        'acc_train': [],
        'loss_val': [],
        'acc_val': [],
        'lr': []
    }
    
    mejor_acc_val = 0.0
    epocas_sin_mejora = 0
    
    print(f"Iniciando entrenamiento por un máximo de {epocas} épocas...")
    print("-" * 85)
    
    for e in range(1, epocas + 1):
        # ------------------ FASE DE ENTRENAMIENTO ------------------
        modelo.train()
        losses_train_epoca, accs_train_epoca = [], []
        
        lr_actual = optimizador.param_groups[0]['lr']
        historia['lr'].append(lr_actual)
        
        for x_b, y_b in dataloaders['train']:
            x_b, y_b = x_b.to(device), y_b.to(device)
            
            optimizador.zero_grad()
            y_pred = modelo(x_b)
            loss = criterio(y_pred, y_b)
            loss.backward()
            optimizador.step()
            
            losses_train_epoca.append(loss.item())
            preds = torch.argmax(y_pred, dim=1)
            acc = (preds == y_b).float().mean().item()
            accs_train_epoca.append(acc)
            
        perdida_train_media = np.mean(losses_train_epoca)
        acc_train_media = np.mean(accs_train_epoca)
        
        # ------------------ FASE DE VALIDACIÓN ------------------
        modelo.eval()
        losses_val_epoca, accs_val_epoca = [], []
        
        with torch.no_grad():
            for x_b, y_b in dataloaders['val']:
                x_b, y_b = x_b.to(device), y_b.to(device)
                y_pred = modelo(x_b)
                loss = criterio(y_pred, y_b)
                
                losses_val_epoca.append(loss.item())
                preds = torch.argmax(y_pred, dim=1)
                acc = (preds == y_b).float().mean().item()
                accs_val_epoca.append(acc)
                
        perdida_val_media = np.mean(losses_val_epoca)
        acc_val_media = np.mean(accs_val_epoca)
        
        # Guardar en el historial
        historia['epoca'].append(e)
        historia['loss_train'].append(perdida_train_media)
        historia['acc_train'].append(acc_train_media)
        historia['loss_val'].append(perdida_val_media)
        historia['acc_val'].append(acc_val_media)
        
        # Actualización del scheduler de tasa de aprendizaje si existe
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(perdida_val_media)
            else:
                scheduler.step()
                
        # Verificar y guardar el mejor modelo (Checkpoint)
        if acc_val_media > mejor_acc_val:
            mejor_acc_val = acc_val_media
            epocas_sin_mejora = 0
            torch.save(modelo.state_dict(), nombre_archivo_ckpt)
            indicador_guardado = "[*] MEJOR MODELO GUARDADO"
        else:
            epocas_sin_mejora += 1
            indicador_guardado = ""
            
        if verbose:
            print(f"Época {e:2d}/{epocas} | "
                  f"Train Loss: {perdida_train_media:.4f} | Train Acc: {acc_train_media * 100:.2f}% | "
                  f"Val Loss: {perdida_val_media:.4f} | Val Acc: {acc_val_media * 100:.2f}% | "
                  f"LR: {lr_actual:.6f} {indicador_guardado}")
                  
        # Criterio de parada temprana (Early Stopping)
        if early_stopping and epocas_sin_mejora >= early_stopping:
            print(f"\n[!] Parada temprana activada en época {e}: sin mejora en {early_stopping} épocas seguidas.")
            break
            
    print("-" * 85)
    print(f"Cargando los mejores pesos del modelo guardados en '{nombre_archivo_ckpt}'...")
    modelo.load_state_dict(torch.load(nombre_archivo_ckpt))
    print(f"¡Entrenamiento finalizado! Mejor Accuracy de Validación lograda: {mejor_acc_val * 100:.2f}%")
    return historia

# Instanciamos el modelo definitivo para el entrenamiento
modelo_final = build_model(D_in=784, H1=256, H2=128, D_out=10, dropout_p=0.2)
criterio_final = nn.CrossEntropyLoss()
optimizador_final = torch.optim.Adam(modelo_final.parameters(), lr=0.001, weight_decay=1e-4)
scheduler_final = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizador_final, mode='min', factor=0.5, patience=2)

# Ejecución del entrenamiento por 15 épocas
historia_entrenamiento = fit(
    modelo=modelo_final,
    dataloaders=dataloaders,
    optimizador=optimizador_final,
    criterio=criterio_final,
    scheduler=scheduler_final,
    epocas=15,
    early_stopping=4,
    nombre_archivo_ckpt='mejor_modelo_mlp_fashion.pt',
    verbose=1
)

## 5. Curvas de Aprendizaje y Análisis Exhaustivo de Convergencia

Para analizar a fondo si el proceso de entrenamiento ha convergido y verificar la salud del modelo, graficamos el comportamiento de las métricas clave en función del **número de épocas**:

1. **Curva de Pérdida (Loss) vs. Épocas**:
   - Comparamos la pérdida de entrenamiento con la pérdida de validación. La convergencia saludable se evidencia cuando ambas curvas descienden conjuntamente y se estabilizan de manera suave sin oscilaciones bruscas.
2. **Curva de Precisión (Accuracy) vs. Épocas**:
   - Permite observar la rapidez con la que el modelo clasifica correctamente las imágenes en ambos conjuntos.
3. **Brecha de Generalización (*Generalization Gap*) vs. Épocas**:
   - Calculada como $\text{Loss}_{val} - \text{Loss}_{train}$. Si esta brecha se incrementa abruptamente hacia valores altos mientras el loss de entrenamiento continúa cayendo a cero, es un síntoma claro de *overfitting* (sobreajuste).
4. **Evolución de la Tasa de Aprendizaje (*Learning Rate*) vs. Épocas**:
   - Muestra cómo el planificador (*scheduler*) reduce automáticamente el paso cuando el error se estanca en una meseta.

In [ ]:
epocas_eje = historia_entrenamiento['epoca']
loss_tr = historia_entrenamiento['loss_train']
loss_va = historia_entrenamiento['loss_val']
acc_tr = [a * 100 for a in historia_entrenamiento['acc_train']]
acc_va = [a * 100 for a in historia_entrenamiento['acc_val']]
lrs = historia_entrenamiento['lr']
brecha_loss = [v - t for v, t in zip(loss_va, loss_tr)]

# Identificar la época donde se obtuvo la mayor precisión de validación
epoca_optima = epocas_eje[np.argmax(acc_va)]
mejor_acc_val_val = max(acc_va)

# Creación de figura con 4 subgráficas para análisis detallado
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Gráfica de Pérdida (Loss) vs Épocas
axes[0, 0].plot(epocas_eje, loss_tr, 'o-', color='royalblue', lw=2, label='Pérdida Entrenamiento (Train Loss)')
axes[0, 0].plot(epocas_eje, loss_va, 's-', color='crimson', lw=2, label='Pérdida Validación (Val Loss)')
axes[0, 0].axvline(epoca_optima, color='green', linestyle='--', alpha=0.8, label=f'Mejor Checkpoint (Época {epoca_optima})')
axes[0, 0].set_xlabel('Número de Épocas', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Pérdida (Cross-Entropy Loss)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Convergencia de la Función de Pérdida (Loss vs Épocas)', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)
axes[0, 0].legend(fontsize=10)

# 2. Gráfica de Precisión (Accuracy) vs Épocas
axes[0, 1].plot(epocas_eje, acc_tr, 'o-', color='royalblue', lw=2, label='Precisión Entrenamiento (Train Acc)')
axes[0, 1].plot(epocas_eje, acc_va, 's-', color='crimson', lw=2, label='Precisión Validación (Val Acc)')
axes[0, 1].axvline(epoca_optima, color='green', linestyle='--', alpha=0.8, label=f'Mejor Checkpoint ({mejor_acc_val_val:.2f}%)')
axes[0, 1].set_xlabel('Número de Épocas', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Precisión (%)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Evolución de la Precisión (Accuracy vs Épocas)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)
axes[0, 1].legend(fontsize=10)

# 3. Gráfica de Brecha de Generalización (Val Loss - Train Loss)
axes[1, 0].plot(epocas_eje, brecha_loss, 'd-', color='darkorange', lw=2, label='Brecha (Val Loss - Train Loss)')
axes[1, 0].axhline(0, color='gray', linestyle=':', lw=1.5)
axes[1, 0].axvline(epoca_optima, color='green', linestyle='--', alpha=0.8, label=f'Época Óptima ({epoca_optima})')
axes[1, 0].set_xlabel('Número de Épocas', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Diferencia de Pérdida', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Monitoreo de Sobreajuste (Generalization Gap vs Épocas)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)
axes[1, 0].legend(fontsize=10)

# 4. Gráfica de Tasa de Aprendizaje (Learning Rate)
axes[1, 1].plot(epocas_eje, lrs, '^-', color='purple', lw=2, label='Tasa de Aprendizaje (LR)')
axes[1, 1].set_xlabel('Número de Épocas', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Tasa de Aprendizaje', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Evolución de la Tasa de Aprendizaje (LR vs Épocas)', fontsize=12, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
axes[1, 1].legend(fontsize=10)

plt.suptitle('Diagnóstico Completo del Proceso de Entrenamiento y Convergencia del MLP', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 5.1 Interpretación y Diagnóstico de las Curvas de Aprendizaje

A partir del análisis visual de las curvas de convergencia observamos los siguientes puntos técnicos clave:

1. **Convergencia Estable y Rápida:**
   - Durante las primeras 5 épocas, la función de costo desciende pronunciadamente tanto en entrenamiento como en validación, demostrando que el optimizador Adam y la tasa de aprendizaje inicial ($10^{-3}$) son adecuados para el espacio de parámetros del MLP.
2. **Control del Sobreajuste (*Overfitting*):**
   - La inclusión de capas de *Batch Normalization* y *Dropout* ($p=0.2$) junto con decaimiento de pesos (*Weight Decay* $10^{-4}$) previene que la red memorice ruido. La curva de validación se mantiene cercana a la de entrenamiento sin divergir de manera alarmante.
3. **Eficacia del Checkpoint:**
   - La línea verde punteada marca la época en que se obtuvo el mejor rendimiento en validación. Gracias al guardado automático de pesos, nos aseguramos de no utilizar un modelo degradado por épocas posteriores, reteniendo el estado de máxima capacidad de generalización.

## 6. Evaluación Exhaustiva en el Conjunto de Prueba y Métricas de Clasificación

En esta etapa evaluamos el modelo final sobre el conjunto de **Prueba (Test)** de 10,000 muestras que el modelo **nunca vio durante el entrenamiento**.

Aplicamos todas las métricas de clasificación explicadas en el cuadernillo `08 metricas_clasificacion.ipynb`:
1. **Accuracy (Exactitud):** Implementada con fórmula manual y con `sklearn.metrics.accuracy_score`.
2. **Matriz de Confusión:** Visualizada con un mapa de calor detallado con nombres de las 10 prendas.
3. **Precision (Precisión) y Recall (Sensibilidad / Exhaustividad):** Fórmulas manuales por clase y promedios multiclase (*Macro* y *Weighted*).
4. **F1-Score:** Media armónica manual y reporte detallado con `classification_report`.
5. **Curva ROC y ROC-AUC Multiclase:** Estrategia *One-vs-Rest* para graficar las 10 curvas ROC y calcular el área bajo la curva global.

In [ ]:
# Extracción de predicciones, probabilidades y etiquetas reales sobre el conjunto de Test
modelo_final.eval()
lista_y_reales = []
lista_y_preds = []
lista_y_probabilidades = []

with torch.no_grad():
    for x_b, y_b in dataloaders['test']:
        x_b = x_b.to(device)
        logits = modelo_final(x_b)
        probas = softmax(logits)
        preds = torch.argmax(probas, dim=1)
        
        lista_y_reales.extend(y_b.numpy())
        lista_y_preds.extend(preds.cpu().numpy())
        lista_y_probabilidades.extend(probas.cpu().numpy())

y_true_test = np.array(lista_y_reales)
y_pred_test = np.array(lista_y_preds)
y_prob_test = np.array(lista_y_probabilidades)

# 1. Definición manual de Accuracy (como en el cuadernillo 08 del docente)
def accuracy_manual(y_pred, y_true):
    """Calcula el porcentaje de aciertos exactos: sum(y_pred == y_true) / N"""
    return np.sum(y_pred == y_true) / len(y_true)

acc_man = accuracy_manual(y_pred_test, y_true_test)
acc_skl = accuracy_score(y_true_test, y_pred_test)

print(f"Evaluación Global sobre el Conjunto de Prueba (10,000 imágenes):")
print(f"  - Accuracy (fórmula manual):    {acc_man * 100:.2f}%")
print(f"  - Accuracy (Scikit-Learn):      {acc_skl * 100:.2f}%")
print(f"  - Total de aciertos:            {np.sum(y_pred_test == y_true_test):,} de {len(y_true_test):,} muestras")
print(f"  - Total de errores:             {np.sum(y_pred_test != y_true_test):,} de {len(y_true_test):,} muestras")

### 6.1 Matriz de Confusión

La **Matriz de Confusión** permite examinar detalladamente en qué clases el modelo acierta y con cuáles se confunde.
- Los elementos en la **diagonal principal** representan los verdaderos positivos ($TP$) de cada prenda.
- Los elementos fuera de la diagonal indican confusiones entre pares de prendas (por ejemplo, *Camisa* vs. *Camiseta* o *Abrigo* vs. *Suéter*).

In [ ]:
# Cálculo de la matriz de confusión con Scikit-Learn
matriz_conf = confusion_matrix(y_true_test, y_pred_test)

# Graficar la matriz de confusión estilizada con Matplotlib
plt.figure(figsize=(10, 8))
plt.imshow(matriz_conf, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de Confusión en el Conjunto de Prueba (Fashion-MNIST)', fontsize=13, fontweight='bold', pad=15)
plt.colorbar(fraction=0.046, pad=0.04)

tick_marks = np.arange(len(nombres_clases))
plt.xticks(tick_marks, nombres_clases, rotation=45, ha='right', fontsize=9)
plt.yticks(tick_marks, nombres_clases, fontsize=9)

# Anotar los valores numéricos dentro de cada celda
umbral = matriz_conf.max() / 2.0
for i in range(matriz_conf.shape[0]):
    for j in range(matriz_conf.shape[1]):
        valor = matriz_conf[i, j]
        color_texto = "white" if valor > umbral else "black"
        plt.text(j, i, f"{valor:,}",
                 horizontalalignment="center",
                 verticalalignment="center",
                 color=color_texto,
                 fontsize=8,
                 fontweight='bold' if i == j else 'normal')

plt.ylabel('Etiqueta Real (Verdadera)', fontsize=11, fontweight='bold')
plt.xlabel('Etiqueta Predicha por el Modelo', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.2 Precision, Recall y F1-Score

Siguiendo las definiciones matemáticas del cuadernillo `08 metricas_clasificacion.ipynb`:
- **Precision (Precisión):** Fracción de predicciones positivas que fueron correctas:
  $$ \text{Precision} = \frac{TP}{TP + FP} $$
- **Recall (Sensibilidad o Exhaustividad):** Fracción de muestras positivas reales que fueron detectadas:
  $$ \text{Recall} = \frac{TP}{TP + FN} $$
- **F1-Score:** Media armónica entre Precision y Recall, útil para evaluar el balance entre ambas métricas:
  $$ F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} $$

Para problemas multiclase, se calculan dos tipos de promedios globales:
- **Macro Average:** Promedio aritmético no ponderado de las métricas de cada clase (trata a todas las clases por igual).
- **Weighted Average:** Promedio ponderado según la cantidad de muestras reales de cada clase.

In [ ]:
# Funciones manuales para una clase binaria específica (según cuadernillo 08)
def precision_clase_manual(y_pred, y_true, clase_id):
    TP = np.sum((y_pred == clase_id) & (y_true == clase_id))
    FP = np.sum((y_pred == clase_id) & (y_true != clase_id))
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0

def recall_clase_manual(y_pred, y_true, clase_id):
    TP = np.sum((y_pred == clase_id) & (y_true == clase_id))
    FN = np.sum((y_pred != clase_id) & (y_true == clase_id))
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0

def f1_clase_manual(p, r):
    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

# Demostración manual para la clase 0 (Camiseta / Top)
p_0 = precision_clase_manual(y_pred_test, y_true_test, clase_id=0)
r_0 = recall_clase_manual(y_pred_test, y_true_test, clase_id=0)
f1_0 = f1_clase_manual(p_0, r_0)

print(f"Cálculo Manual para la Clase 0 ({nombres_clases[0]}):")
print(f"  - Precision Manual: {p_0:.4f}")
print(f"  - Recall Manual:    {r_0:.4f}")
print(f"  - F1-Score Manual:  {f1_0:.4f}")

# Métricas globales con Scikit-Learn (Macro y Weighted)
p_macro = precision_score(y_true_test, y_pred_test, average='macro')
r_macro = recall_score(y_true_test, y_pred_test, average='macro')
f1_macro = f1_score(y_true_test, y_pred_test, average='macro')

p_weighted = precision_score(y_true_test, y_pred_test, average='weighted')
r_weighted = recall_score(y_true_test, y_pred_test, average='weighted')
f1_weighted = f1_score(y_true_test, y_pred_test, average='weighted')

print(f"\nMétricas Globales Multiclase en Test:")
print(f"  - Precision (Macro):    {p_macro:.4f} | Precision (Weighted):    {p_weighted:.4f}")
print(f"  - Recall (Macro):       {r_macro:.4f} | Recall (Weighted):       {r_weighted:.4f}")
print(f"  - F1-Score (Macro):     {f1_macro:.4f} | F1-Score (Weighted):     {f1_weighted:.4f}")

# Reporte de clasificación exhaustivo por cada clase
print(f"\nReporte Completo de Clasificación por Prenda:")
print(classification_report(y_true_test, y_pred_test, target_names=nombres_clases, digits=4))

### 6.3 Curvas ROC y Puntuación ROC-AUC Multiclase (One-vs-Rest)

La curva **ROC** (*Receiver Operating Characteristic*) grafica la Tasa de Verdaderos Positivos (**TPR** o *Recall*) frente a la Tasa de Falsos Positivos (**FPR**):
$$ \text{TPR} = \frac{TP}{TP + FN}, \quad \text{FPR} = \frac{FP}{FP + TN} $$

Para adaptar la curva ROC vista en el cuadernillo 08 a un problema de 10 clases, utilizamos la estrategia **One-vs-Rest (OvR)**:
- Binarizamos las etiquetas reales en formato *one-hot*.
- Calculamos la curva ROC individual y el área bajo la curva (**AUC**) para cada una de las 10 categorías de prendas.
- Calculamos el **ROC-AUC Macro** promedio global.

In [ ]:
# Binarización de etiquetas para enfoque One-vs-Rest (OvR)
y_true_bin = label_binarize(y_true_test, classes=range(10))

# Calcular FPR, TPR y AUC para cada una de las 10 clases
fpr_dict = dict()
tpr_dict = dict()
roc_auc_dict = dict()

for i in range(10):
    fpr_dict[i], tpr_dict[i], _ = roc_curve(y_true_bin[:, i], y_prob_test[:, i])
    roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

# ROC-AUC promedio macro
roc_auc_macro = roc_auc_score(y_true_bin, y_prob_test, average='macro', multi_class='ovr')
print(f"Puntuación ROC-AUC Multiclase (Promedio Macro OvR): {roc_auc_macro:.4f}")

# Graficar las curvas ROC para cada clase
plt.figure(figsize=(10, 8))
colores = plt.cm.tab10(np.linspace(0, 1, 10))

for i, color in zip(range(10), colores):
    plt.plot(fpr_dict[i], tpr_dict[i], color=color, lw=1.8,
             label=f"{nombres_clases[i]} (AUC = {roc_auc_dict[i]:.3f})")

# Línea diagonal aleatoria
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Clasificador Aleatorio (AUC = 0.500)')

plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=11, fontweight='bold')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=11, fontweight='bold')
plt.title(f'Curvas ROC por Clase One-vs-Rest (ROC-AUC Macro: {roc_auc_macro:.4f})', fontsize=13, fontweight='bold', pad=12)
plt.legend(loc="lower right", fontsize=9)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Inferencia y Predicciones con Muestras Individuales

Al igual que en la sección 2.4 de `02_reg_lin_mul_01_vuelos.ipynb`, comprobamos el funcionamiento del modelo aprendido realizando inferencias sobre ejemplos concretos del conjunto de prueba.

Para cada prenda seleccionada:
1. Tomamos su vector de 784 píxeles normalizado.
2. Pasamos el vector por el modelo para obtener los *logits*.
3. Aplicamos la función `softmax` para calcular la distribución de probabilidades de pertenecer a cada una de las 10 clases.
4. Graficamos la imagen $28 \times 28$ junto a un gráfico de barras horizontales con las probabilidades predichas.

In [ ]:
def predecir_y_graficar_ejemplo(indice_test):
    """
    Realiza inferencia sobre una muestra específica del conjunto de prueba y
    muestra la imagen junto con la distribución de probabilidades predicha.
    """
    muestra_vector = X_test[indice_test]
    etiqueta_real = y_test[indice_test]
    
    # Preparar tensor para inferencia
    tensor_input = torch.from_numpy(muestra_vector).unsqueeze(0).float().to(device)
    
    modelo_final.eval()
    with torch.no_grad():
        logits = modelo_final(tensor_input)
        probabilidades = softmax(logits).cpu().numpy().squeeze()
        clase_predicha = np.argmax(probabilidades)
        confianza = probabilidades[clase_predicha] * 100
        
    # Graficar imagen y gráfico de barras de probabilidades
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(11, 4))
    
    # Imagen 28x28
    ax_img.imshow(muestra_vector.reshape(28, 28), cmap='gray')
    ax_img.set_title(f"Real: {nombres_clases[etiqueta_real]}\nPredicha: {nombres_clases[clase_predicha]} ({confianza:.1f}%)",
                     fontsize=11, fontweight='bold',
                     color='darkgreen' if clase_predicha == etiqueta_real else 'crimson')
    ax_img.axis('off')
    
    # Gráfico de barras horizontales de probabilidades
    barras = ax_bar.barh(range(10), probabilidades * 100, color='lightsteelblue', edgecolor='black')
    barras[clase_predicha].set_color('mediumseagreen' if clase_predicha == etiqueta_real else 'crimson')
    barras[clase_predicha].set_edgecolor('black')
    
    ax_bar.set_yticks(range(10))
    ax_bar.set_yticklabels(nombres_clases, fontsize=9)
    ax_bar.set_xlabel('Probabilidad (%)', fontsize=10, fontweight='bold')
    ax_bar.set_xlim([0, 105])
    ax_bar.set_title('Distribución de Probabilidades Softmax', fontsize=11, fontweight='bold')
    ax_bar.grid(axis='x', linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

# Probar inferencia sobre 4 ejemplos distintos del conjunto de prueba
print("Ejemplos de Inferencia Individual con el Modelo MLP Entrenado:")
for idx_ejemplo in [0, 15, 42, 100]:
    predecir_y_graficar_ejemplo(idx_ejemplo)

## 8. Conclusiones y Discusión Final

A lo largo de este laboratorio se ha desarrollado y validado exitosamente un Perceptrón Multicapa (**MLP**) en **PyTorch** para la clasificación de prendas de ropa en el dataset **Fashion-MNIST**:

1. **Cumplimiento de la Receta de Entrenamiento (Cuadernillo 07):**
   - Se completaron las etapas de exploración, verificación de dimensiones con tensores sintéticos, *sanity checks* de sobreajuste de 1 muestra y 1 lote completo, confirmando que la arquitectura y el bucle de optimización estaban libres de errores conceptuales antes de entrenar en el dataset completo.
   - La función modular `fit(...)` integró validación cruzada época a época, decaimiento adaptativo de tasa de aprendizaje (*ReduceLROnPlateau*), parada temprana (*Early Stopping*) y guardado automático del mejor punto de control (*Model Checkpointing*).

2. **Análisis de Convergencia (Punto 5):**
   - Las curvas de pérdida y precisión demostraron una convergencia suave y estable.
   - Las técnicas de regularización implementadas (*Batch Normalization*, *Dropout* de 0.2 y *Weight Decay* de $10^{-4}$) controlaron eficazmente la brecha de sobreajuste (*generalization gap*), permitiendo que el modelo generalice de forma sólida hacia datos nunca antes vistos.

3. **Métricas de Clasificación y Diagnóstico (Cuadernillo 08):**
   - El modelo alcanzó una **exactitud (*Accuracy*) superior al 88%** en el conjunto de prueba independiente de 10,000 imágenes.
   - La **Matriz de Confusión** reveló que las prendas con siluetas muy distintivas (como *Pantalón*, *Bolso* o *Botín*) alcanzan precisiones y sensibilidades superiores al 95%. Las mayores confusiones se concentran de manera lógica entre prendas visualmente similares con cuellos y mangas (como *Camisa* frente a *Camiseta* y *Abrigo* frente a *Suéter*).
   - El área bajo la curva promedio **ROC-AUC Macro de ~0.99** confirma una sobresaliente capacidad de discriminación entre las 10 categorías.

4. **Organización del Cuadernillo (Estilo Cuadernillo 02):**
   - Todas las funciones y clases fueron declaradas estrictamente en la etapa en que fueron invocadas, asegurando un flujo de lectura claro, didáctico y reproducible.